# ExoHunter Verification Protocol

**12 tasks to verify that the pipeline produces correct results**

Run each cell in order. Every task ends with a PASS/FAIL assertion.
All 12 must pass.

See `VERIFICATION_GUIDE.md` for detailed explanations of each task.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
from pathlib import Path

results = {}  # track pass/fail for summary
print("Ready.")

---

## Task 1: Installation — test suite passes

In [ ]:
import subprocess, sys

r = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-q", "--tb=no"],
    capture_output=True, text=True, cwd=str(Path.cwd().parent),
)

# Extract pass count from last line
last_line = r.stdout.strip().split("\n")[-1]
print(last_line)

passed = "passed" in last_line and "failed" not in last_line
results["T1_tests"] = passed
print(f"\n{'PASS' if passed else 'FAIL'}: test suite")

---

## Task 2: Download + cache roundtrip

In [ ]:
from exohunter.ingestion.downloader import download_lightcurve
from exohunter.ingestion.cache import load_from_cache, save_to_cache
from exohunter import config

lc = download_lightcurve("TIC 150428135")

time_orig = np.array(lc.time.value, dtype=np.float64)
flux_orig = np.array(lc.flux.value, dtype=np.float64)
print(f"Cadences: {len(time_orig)}")
print(f"Time span: {time_orig[-1] - time_orig[0]:.1f} days")

# Force cache roundtrip
save_to_cache(lc, "TIC 150428135", config.CACHE_DIR)
lc_cached = load_from_cache("TIC 150428135", config.CACHE_DIR)
time_cached = np.array(lc_cached.time.value, dtype=np.float64)
flux_cached = np.array(lc_cached.flux.value, dtype=np.float64)

ok_len = len(time_cached) == len(time_orig)
ok_time = np.allclose(time_cached, time_orig, atol=1e-10)
ok_flux = np.allclose(flux_cached, flux_orig, atol=1e-10)
passed = ok_len and ok_time and ok_flux

results["T2_cache"] = passed
print(f"Length match: {ok_len}")
print(f"Time match:  {ok_time}")
print(f"Flux match:  {ok_flux}")
print(f"\n{'PASS' if passed else 'FAIL'}: cache roundtrip")

---

## Task 3: Preprocessing preserves transit

In [ ]:
from exohunter.preprocessing.pipeline import preprocess_single

processed = preprocess_single(lc, tic_id="TIC 150428135")

fraction_kept = len(processed.time) / len(lc)
print(f"Cadences kept: {fraction_kept:.1%}")

# Phase-fold at TOI-700 b
period = 9.977
phase = (processed.time % period) / period
near = np.abs(phase - 0.5) < 0.015
far = (phase < 0.4) | (phase > 0.6)

ok_retention = fraction_kept > 0.90

if np.sum(near) > 0 and np.sum(far) > 0:
    depth = np.median(processed.flux[far]) - np.median(processed.flux[near])
    print(f"Measured depth at P={period}d: {depth*1e6:.0f} ppm")
    ok_signal = depth > 0.0001
else:
    ok_signal = False
    print("Not enough cadences for phase-fold check")

passed = ok_retention and ok_signal
results["T3_preprocess"] = passed
print(f"\n{'PASS' if passed else 'FAIL'}: preprocessing")

---

## Task 4: BLS detects correct period

In [ ]:
from exohunter.detection.bls import run_bls_lightkurve

lc_for_bls = processed.to_lightcurve()
candidate = run_bls_lightkurve(
    lc_for_bls, tic_id="TIC 150428135",
    min_period=0.5, max_period=45.0, num_periods=15000,
)

print(f"Detected period: {candidate.period:.4f} d")
print(f"Detected depth:  {candidate.depth*100:.4f}%")

known_periods = [9.977, 16.051, 37.426]
match = any(abs(candidate.period - kp) < 0.5 for kp in known_periods)
harmonic = any(
    abs(candidate.period / kp - h) < 0.1
    for kp in known_periods for h in [0.5, 2.0]
)

passed = match or harmonic
results["T4_bls"] = passed
status = "direct match" if match else ("harmonic" if harmonic else "no match")
print(f"\n{'PASS' if passed else 'FAIL'}: BLS detection ({status})")

---

## Task 5: Numba and lightkurve BLS agree

In [ ]:
from exohunter.detection.bls import run_bls_numba, _NUMBA_AVAILABLE
from tests.conftest import make_synthetic_transit_lc

syn_lc = make_synthetic_transit_lc(period=7.0, depth=0.01, duration=0.15,
                                    noise=0.0005, n_points=15000)

lk = run_bls_lightkurve(syn_lc, tic_id="CMP", min_period=2.0,
                         max_period=12.0, num_periods=5000)
print(f"lightkurve: P = {lk.period:.4f} d")

ok_lk = abs(lk.period - 7.0) < 0.1

if _NUMBA_AVAILABLE:
    nb = run_bls_numba(syn_lc.time.value, syn_lc.flux.value, tic_id="CMP",
                        min_period=2.0, max_period=12.0, num_periods=5000)
    print(f"Numba:      P = {nb.period:.4f} d")
    diff = abs(lk.period - nb.period)
    print(f"Difference: {diff:.4f} d")
    ok_agree = diff < 0.2
else:
    print("Numba not available — skipping comparison")
    ok_agree = True

passed = ok_lk and ok_agree
results["T5_numba"] = passed
print(f"\n{'PASS' if passed else 'FAIL'}: BLS implementations")

---

## Task 6: Multi-planet search

In [ ]:
from exohunter.detection.bls import run_iterative_bls

candidates = run_iterative_bls(
    lc_for_bls, tic_id="TIC 150428135",
    min_period=0.5, max_period=45.0, num_periods=15000,
    max_planets=5, min_snr=0.0,
)

known = {"b": 9.977, "c": 16.051, "d": 37.426}
matched = set()

print(f"Found {len(candidates)} candidate(s):\n")
for i, c in enumerate(candidates):
    letter = chr(ord("b") + i)
    tag = ""
    for name, per in known.items():
        if abs(c.period - per) < 1.0:
            tag = f"  <- TOI-700 {name}"
            matched.add(name)
    print(f"  {letter}: P={c.period:.4f} d, depth={c.depth*100:.4f}%{tag}")

passed = len(candidates) >= 2
results["T6_multi"] = passed
print(f"\nMatched: {len(matched)}/3 known planets")
print(f"{'PASS' if passed else 'FAIL'}: multi-planet search")

---

## Task 7: Validation accepts/rejects correctly

In [ ]:
from exohunter.detection.bls import TransitCandidate
from exohunter.detection.validator import validate_candidate

good = TransitCandidate(tic_id="GOOD", period=10.0, epoch=5.0, duration=0.2,
                         depth=0.01, snr=15.0, bls_power=0.5, n_transits=9)
good_v = validate_candidate(good)

bad = TransitCandidate(tic_id="BAD", period=10.0, epoch=5.0, duration=3.0,
                        depth=0.10, snr=2.0, bls_power=0.1, n_transits=1)
bad_v = validate_candidate(bad)

print("Good candidate:")
for t, p in good_v.tests.items():
    print(f"  {'PASS' if p else 'FAIL'} — {t}")
print(f"  Overall: {'VALID' if good_v.is_valid else 'REJECTED'}")

print("\nBad candidate:")
for t, p in bad_v.tests.items():
    print(f"  {'PASS' if p else 'FAIL'} — {t}")
print(f"  Overall: {'VALID' if bad_v.is_valid else 'REJECTED'}")

passed = good_v.is_valid and not bad_v.is_valid
results["T7_valid"] = passed
print(f"\n{'PASS' if passed else 'FAIL'}: validation")

---

## Task 8: Cross-matching — all 4 classes

In [ ]:
from exohunter.catalog.crossmatch import crossmatch_candidate, MatchClass

tests_8 = [
    ("KNOWN_MATCH",    "TIC 150428135", 9.977,  MatchClass.KNOWN_MATCH),
    ("HARMONIC",       "TIC 150428135", 19.95,  MatchClass.HARMONIC),
    ("KNOWN_TOI",      "TIC 150428135", 6.0,    MatchClass.KNOWN_TOI),
    ("NEW_CANDIDATE",  "TIC 999999999", 5.0,    MatchClass.NEW_CANDIDATE),
]

all_ok = True
for label, tic, period, expected in tests_8:
    c = TransitCandidate(tic_id=tic, period=period, epoch=0,
                          duration=0.1, depth=0.001, snr=10, bls_power=1)
    r = crossmatch_candidate(c)
    ok = r.match_class == expected
    all_ok &= ok
    status = "PASS" if ok else "FAIL"
    print(f"  {status}: {label} (P={period}) -> {r.match_class.value}")

results["T8_xmatch"] = all_ok
print(f"\n{'PASS' if all_ok else 'FAIL'}: cross-matching")

---

## Task 9: ML classification on synthetic data

In [ ]:
from exohunter.classification.model import build_pipeline, classify_candidates
from exohunter.classification.features import FEATURE_COLUMNS

rng = np.random.default_rng(42)
rows = []
for label, depth, snr in [("planet", 0.001, 20),
                           ("eclipsing_binary", 0.03, 40),
                           ("false_positive", 0.0001, 3)]:
    for _ in range(200):
        rows.append({
            "period": rng.uniform(1, 15),
            "depth": abs(rng.normal(depth, depth*0.3)),
            "duration": rng.uniform(0.02, 0.15),
            "snr": abs(rng.normal(snr, snr*0.3)),
            "impact_param": rng.uniform(0, 0.8),
            "stellar_teff": rng.normal(5500, 500),
            "stellar_logg": rng.normal(4.3, 0.3),
            "stellar_radius": rng.normal(1.0, 0.3),
            "duration_period_ratio": 0, "depth_log": 0,
            "label": label,
        })

df = pd.DataFrame(rows)
df["duration_period_ratio"] = df["duration"] / df["period"]
df["depth_log"] = np.log10(df["depth"].clip(lower=1e-10))

pipeline = build_pipeline(n_estimators=100, max_depth=10)
pipeline.fit(df[FEATURE_COLUMNS].values, df["label"].values)

ml_results = classify_candidates(pipeline, df)
accuracy = np.mean(df["label"].values == ml_results["ml_class"].values)
print(f"Accuracy: {accuracy:.1%}")

prob_cols = ["ml_prob_planet", "ml_prob_eb", "ml_prob_fp"]
prob_sums = ml_results[prob_cols].sum(axis=1)
ok_probs = np.allclose(prob_sums, 1.0, atol=1e-5)
print(f"Probabilities sum to 1: {ok_probs}")

passed = accuracy > 0.7 and ok_probs
results["T9_ml"] = passed
print(f"\n{'PASS' if passed else 'FAIL'}: ML classification")

---

## Task 10: Export roundtrip (CSV, FITS, VOTable)

In [ ]:
from exohunter.catalog.candidates import CandidateCatalog
from exohunter.catalog.export import export_to_csv, export_to_fits, export_to_votable
from exohunter.detection.validator import ValidationResult
from astropy.table import Table

cat = CandidateCatalog()
c = TransitCandidate(tic_id="TIC 150428135", period=9.977, epoch=2.5,
                      duration=0.095, depth=0.00058, snr=12.4, bls_power=1.0,
                      n_transits=35, name="TOI-700 b")
v = ValidationResult(is_valid=True, tests={"v_shape": True}, flags=[])
cat.add(c, v)

tmp = Path("/tmp/exohunter_export_verify")
tmp.mkdir(exist_ok=True)

# CSV
csv_df = pd.read_csv(export_to_csv(cat, tmp / "test.csv"))
ok_csv = abs(csv_df.iloc[0]["period"] - 9.977) < 0.001
print(f"CSV roundtrip:    {'PASS' if ok_csv else 'FAIL'}")

# FITS
t_fits = Table.read(export_to_fits(cat, tmp / "test.fits"), format="fits")
ok_fits = abs(float(t_fits["period"][0]) - 9.977) < 0.001
print(f"FITS roundtrip:   {'PASS' if ok_fits else 'FAIL'}")

# VOTable
t_vot = Table.read(export_to_votable(cat, tmp / "test.vot.xml"), format="votable")
ok_vot_val = abs(float(t_vot["period"][0]) - 9.977) < 0.001
ok_vot_ucd = t_vot["period"].meta.get("ucd") == "time.period"
ok_vot_unit = str(t_vot["period"].unit) == "d"
ok_vot = ok_vot_val and ok_vot_ucd and ok_vot_unit
print(f"VOTable roundtrip: {'PASS' if ok_vot else 'FAIL'} (value={ok_vot_val}, UCD={ok_vot_ucd}, unit={ok_vot_unit})")

passed = ok_csv and ok_fits and ok_vot
results["T10_export"] = passed
print(f"\n{'PASS' if passed else 'FAIL'}: export roundtrip")

---

## Task 11: Dashboard serves correctly

In [ ]:
import threading, time, urllib.request, json
from scripts.run_dashboard import generate_demo_data
from exohunter.dashboard.app import create_app

data = generate_demo_data()
app = create_app(pipeline_data=data)

server = threading.Thread(
    target=lambda: app.run(host="127.0.0.1", port=9998, debug=False),
    daemon=True,
)
server.start()
time.sleep(4)

try:
    resp = urllib.request.urlopen("http://127.0.0.1:9998/_dash-layout")
    layout = json.dumps(json.loads(resp.read().decode()))

    required = [
        "pipeline-data", "sky-map", "lightcurve-plot", "phase-plot",
        "periodogram-plot", "odd-even-plot", "candidate-table",
        "candidate-selector", "data-source-selector", "xmatch-filter",
        "new-candidates-panel", "cache-stats-body", "ml-status-body",
        "batch-results-body", "reports-gallery-body", "alerts-feed-body",
    ]
    missing = [c for c in required if c not in layout]
    ok_components = len(missing) == 0
    ok_data = "TOI-700" in layout

    if missing:
        print(f"Missing: {missing}")
    print(f"Components ({len(required) - len(missing)}/{len(required)}): {'PASS' if ok_components else 'FAIL'}")
    print(f"Demo data:  {'PASS' if ok_data else 'FAIL'}")

    passed = ok_components and ok_data
except Exception as e:
    print(f"FAIL: could not connect — {e}")
    passed = False

results["T11_dashboard"] = passed
print(f"\n{'PASS' if passed else 'FAIL'}: dashboard")

---

## Task 12: Alert system triggers correctly

In [ ]:
from exohunter.alerts import check_and_dispatch_alerts
import exohunter.config as cfg

summary = pd.DataFrame([
    {"tic_id": "TIC 900000001", "xmatch_class": "NEW_CANDIDATE",
     "snr": 12.0, "is_valid": True, "period": 5.0, "depth": 0.005},
    {"tic_id": "TIC 900000002", "xmatch_class": "KNOWN_MATCH",
     "snr": 15.0, "is_valid": True, "period": 3.0, "depth": 0.01},
    {"tic_id": "TIC 900000003", "xmatch_class": "NEW_CANDIDATE",
     "snr": 3.0, "is_valid": False, "period": 8.0, "depth": 0.001},
])

tmp_alerts = Path("/tmp/exohunter_alert_verify")
tmp_alerts.mkdir(exist_ok=True)
orig = cfg.ALERTS_DIR
cfg.ALERTS_DIR = tmp_alerts

try:
    n = check_and_dispatch_alerts(summary, sector=99)
    alert_files = list(tmp_alerts.glob("alert_*.json"))

    ok_count = n == 1
    ok_file = len(alert_files) >= 1

    if ok_file:
        with open(alert_files[-1]) as f:
            payload = json.load(f)
        ok_content = (payload["n_candidates"] == 1 and
                      payload["candidates"][0]["tic_id"] == "TIC 900000001")
    else:
        ok_content = False

    print(f"Alert count (expect 1): {n} — {'PASS' if ok_count else 'FAIL'}")
    print(f"File created:           {'PASS' if ok_file else 'FAIL'}")
    print(f"Correct content:        {'PASS' if ok_content else 'FAIL'}")

    passed = ok_count and ok_file and ok_content
finally:
    cfg.ALERTS_DIR = orig

results["T12_alerts"] = passed
print(f"\n{'PASS' if passed else 'FAIL'}: alert system")

---

## Final Report

In [ ]:
print("=" * 50)
print("  ExoHunter Verification Report")
print("=" * 50)

all_pass = True
for task_id, passed in sorted(results.items()):
    symbol = "PASS" if passed else "FAIL"
    color = "\033[92m" if passed else "\033[91m"
    reset = "\033[0m"
    print(f"  {color}{symbol}{reset}  {task_id}")
    if not passed:
        all_pass = False

print("=" * 50)
n_pass = sum(1 for v in results.values() if v)
n_total = len(results)
print(f"  {n_pass}/{n_total} tasks passed")

if all_pass:
    print("\n  All verification tasks PASSED.")
    print("  The ExoHunter pipeline is verified correct.")
else:
    failed = [k for k, v in results.items() if not v]
    print(f"\n  FAILED tasks: {', '.join(failed)}")
    print("  Review the output above for details.")